# 02b — Estadísticas descriptivas por clase
*Caracterización completa de las 9 clases acústicas detectadas.*

Cuatro bloques:
1. **Confianza por clase** — distribución, media, P25/P75 y diferencia por fuente.
2. **Duración de evento** — distribución de `t_end−t_start` por clase.
3. **Tasa de detección** — eventos/min y eventos/km por clase, trayecto y fuente.
4. **Co-ocurrencia** — qué clases aparecen juntas en ventana ±5 s (matriz y red).

> Las clases fuera de dominio se incluyen descriptivamente; el peligro se restringe a Horn(0)+Siren(1) como se argumenta en `02_reliability_and_classes`.

In [ ]:
import sys, warnings
sys.path.insert(0, "../scripts")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geo_utils as gu

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.width", 140)

geo = gu.load_geo()
geo = geo[~geo["trayecto"].isin(gu.BAD_TRAYECTOS)].copy()
geo["class_name"] = geo["class"].map(gu.CLASS_NAMES)
geo["duration_s"] = (geo.t_end - geo.t_start).dt.total_seconds()

tracks = gu.load_tracks()
tracks = tracks[~tracks["trayecto"].isin(gu.BAD_TRAYECTOS)].copy()

CLASS_ORDER = [gu.CLASS_NAMES[i] for i in range(9)]
FOCUS_ORDER = ["Horn", "Siren", "Speech"]   # clases de interés principal
ROAD_NAMES  = [gu.CLASS_NAMES[c] for c in gu.ROAD_CLASSES]

print(f"{len(geo):,} detecciones | {geo.trayecto.nunique()} trayectos | fuentes: {geo.source.unique().tolist()}")

In [ ]:
# Paleta consistente clase → color
_pal_list = sns.color_palette("tab10", 9)
CLASS_PALETTE = {gu.CLASS_NAMES[i]: _pal_list[i] for i in range(9)}
DOMAIN_COLOR = {"Vial (peligro)": "#c0392b", "Contexto": "#2980b9", "Fuera de dominio": "#95a5a6"}
def domain_color(name):
    if name in ("Horn","Siren"):     return DOMAIN_COLOR["Vial (peligro)"]
    if name == "Speech":             return DOMAIN_COLOR["Contexto"]
    return DOMAIN_COLOR["Fuera de dominio"]

## 1 — Confianza por clase
¿El modelo asigna confianzas similares a todas las clases o hay clases sistemáticamente más inciertas? Diferencias entre `mic` y `mobile`.

In [ ]:
# 1a — Tabla resumen de confianza por clase
conf_tab = (geo.groupby("class_name")["confidence"]
              .agg(n="count", mean="mean", std="std",
                   p25=lambda s: s.quantile(.25),
                   median="median",
                   p75=lambda s: s.quantile(.75))
              .loc[CLASS_ORDER].round(3))
display(conf_tab)

In [ ]:
# 1b — Violines de confianza por clase (todas, color=dominio)
fig, ax = plt.subplots(figsize=(13, 5))
palette_class = {n: domain_color(n) for n in CLASS_ORDER}
vio = sns.violinplot(data=geo, x="class_name", y="confidence",
                     order=CLASS_ORDER, palette=palette_class, cut=0, ax=ax, linewidth=.8)
ax.axhline(0.80, color="#c0392b", ls="--", lw=1.5, label="umbral 0.80")
ax.axhline(0.10, color="#888", ls=":", lw=1,   label="umbral mínimo 0.10")
ax.set_title("Distribución de confianza por clase"); ax.set_xlabel("")
ax.set_ylabel("confianza"); ax.legend(fontsize=12)
plt.xticks(rotation=35, ha="right"); plt.tight_layout()
plt.savefig("../outputs/stat_conf_violin.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# 1c — Confianza por clase y fuente (strip + box)
fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=geo, x="class_name", y="confidence", hue="source",
            order=CLASS_ORDER, palette="Set2", ax=ax, linewidth=.9, fliersize=2)
ax.axhline(0.80, color="#c0392b", ls="--", lw=1.5, label="umbral 0.80")
ax.set_title("Confianza por clase y fuente (mic vs mobile)")
ax.set_xlabel(""); ax.set_ylabel("confianza"); ax.legend(title="fuente")
plt.xticks(rotation=35, ha="right"); plt.tight_layout()
plt.savefig("../outputs/stat_conf_by_source.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# 1d — Distribución de confianza Horn+Siren (las que importan)
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
for ax, cid in zip(axes, [0, 1]):
    sub = geo[geo["class"] == cid]
    for src, grp in sub.groupby("source"):
        ax.hist(grp.confidence, bins=20, alpha=.6, label=src, edgecolor="white")
    ax.axvline(0.80, color="#c0392b", ls="--", lw=1.5)
    ax.set_title(f"{gu.CLASS_NAMES[cid]}  (n={len(sub)})")
    ax.set_xlabel("confianza"); ax.legend(title="fuente")
plt.suptitle("Horn y Siren — distribución de confianza detallada")
plt.tight_layout(); plt.savefig("../outputs/stat_conf_horn_siren.png", dpi=130, bbox_inches="tight"); plt.show()

**Lectura.** Clases con confianza sistemáticamente alta pueden estar sobredispararando (modelo muy seguro en OOD). Horn y Siren con confianza muy baja y skewed → umbral 0.80 deja muy pocos eventos (ver NB-02 sensibilidad). Diferencias mic/mobile revelan sesgo de sensor.

## 2 — Duración de evento por clase
Cada detección tiene un intervalo temporal [`t_start`, `t_end`] ≤ 7.5 s (tamaño del chunk). La duración refleja cuánto tiempo el modelo sostiene la detección dentro del espectrograma.

In [ ]:
dur_tab = (geo.groupby("class_name")["duration_s"]
              .agg(n="count", mean="mean", std="std",
                   p25=lambda s: s.quantile(.25),
                   median="median",
                   p75=lambda s: s.quantile(.75),
                   max="max")
              .loc[CLASS_ORDER].round(3))
display(dur_tab)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
sns.violinplot(data=geo, x="class_name", y="duration_s",
               order=CLASS_ORDER, palette=palette_class, cut=0, ax=ax, linewidth=.8)
ax.set_title("Duración de evento por clase"); ax.set_xlabel(""); ax.set_ylabel("duración (s)")
plt.xticks(rotation=35, ha="right"); plt.tight_layout()
plt.savefig("../outputs/stat_duration_violin.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# Duración vs confianza (scatter Horn+Siren — buscar correlación)
fig, ax = plt.subplots(figsize=(8, 5))
for cid in [0, 1]:
    sub = geo[geo["class"] == cid]
    ax.scatter(sub.confidence, sub.duration_s, alpha=.4, s=20,
               label=gu.CLASS_NAMES[cid], color=domain_color(gu.CLASS_NAMES[cid]))
ax.set_xlabel("confianza"); ax.set_ylabel("duración (s)")
ax.set_title("Confianza vs duración — Horn y Siren")
ax.legend(); plt.tight_layout()
plt.savefig("../outputs/stat_conf_vs_duration.png", dpi=130, bbox_inches="tight"); plt.show()
# Correlación de Spearman (no asume normalidad)
from scipy.stats import spearmanr
for cid in [0, 1]:
    sub = geo[geo["class"] == cid].dropna(subset=["confidence","duration_s"])
    r, p = spearmanr(sub.confidence, sub.duration_s)
    print(f"{gu.CLASS_NAMES[cid]}: Spearman r={r:.3f}  p={p:.4f}  n={len(sub)}")

**Lectura.** Duración corta + baja confianza en Horn/Siren → pueden ser eventos reales breves (bocinazo corto) o FP. Una correlación positiva confianza↔duración sugiere que las detecciones más largas son más fiables (el modelo 've' el evento en más frames del espectrograma).

## 3 — Tasa de detección: eventos/min y eventos/km
Normaliza el recuento de detecciones por el tiempo grabado y la distancia recorrida en cada trayecto. Permite comparar trayectos de distinta longitud y duración.

In [ ]:
# Duración por trayecto (min)
dur_tr = (geo.groupby("trayecto")
             .agg(t0=("t_start","min"), t1=("t_start","max"))
             .assign(minutes=lambda d:(d.t1-d.t0).dt.total_seconds()/60)
             ["minutes"])

# Distancia por trayecto (km) — haversine acumulada sobre trackpoints
import sys; sys.path.insert(0, "../scripts") if "../scripts" not in sys.path else None
from geo_utils import haversine_m
dist_rows = []
for tr, grp in tracks.sort_values(["trayecto","time"]).groupby("trayecto"):
    lats, lons = grp.lat.values, grp.lon.values
    d = haversine_m(lats[:-1], lons[:-1], lats[1:], lons[1:]).sum() / 1000
    dist_rows.append({"trayecto": tr, "km": d})
dist_tr = pd.DataFrame(dist_rows).set_index("trayecto")["km"]
print("Distancia media por trayecto:", round(dist_tr.mean(),2), "km")

In [ ]:
# Tasa por clase y trayecto
rate_rows = []
for (tr, cn), grp in geo.groupby(["trayecto","class_name"]):
    n = len(grp)
    mins = dur_tr.get(tr, np.nan)
    km   = dist_tr.get(tr, np.nan)
    rate_rows.append({"trayecto": tr, "class_name": cn, "n": n,
                      "per_min": n/mins if mins else np.nan,
                      "per_km":  n/km   if km   else np.nan,
                      "source": grp.source.iloc[0]})
rates = pd.DataFrame(rate_rows)

# Resumen por clase
rate_sum = (rates.groupby("class_name")[["per_min","per_km"]]
                 .agg(["mean","std"]).loc[CLASS_ORDER].round(3))
display(rate_sum)

In [ ]:
# Boxplot eventos/min por clase
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, col, ylabel in zip(axes, ["per_min","per_km"], ["eventos / min","eventos / km"]):
    sns.boxplot(data=rates, x="class_name", y=col, hue="source",
                order=CLASS_ORDER, palette="Set2", ax=ax, linewidth=.9, fliersize=2)
    ax.set_title(f"Tasa de detección — {ylabel}"); ax.set_xlabel(""); ax.set_ylabel(ylabel)
    ax.legend(title="fuente"); plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
plt.tight_layout()
plt.savefig("../outputs/stat_rate_per_class.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# Top-10 trayectos por tasa de Horn+Siren combinada (/km)
top = (rates[rates.class_name.isin(ROAD_NAMES)]
         .groupby(["trayecto","source"])["per_km"].sum()
         .reset_index().sort_values("per_km", ascending=False).head(10))
display(top)

**Lectura.** Tasa alta de Horn/km en ciertos trayectos identifica corredores más conflictivos independientemente de su longitud. Diferencia mic/mobile en tasa → sesgo de sensor cuantificado.

## 4 — Co-ocurrencia entre clases (ventana ±5 s)
Dos detecciones co-ocurren si sus ventanas temporales se solapan ± 5 s dentro del mismo trayecto. La matriz muestra con qué frecuencia cada par de clases aparece simultáneamente.

In [ ]:
WINDOW_S = 5.0
pairs = []
for tr, grp in geo.groupby("trayecto"):
    grp = grp.sort_values("t_start").reset_index(drop=True)
    ts  = grp.t_start.values.astype("datetime64[us]").astype(np.int64) / 1e6  # → segundos float
    cls = grp["class"].values
    for i in range(len(grp)):
        for j in range(i+1, len(grp)):
            if abs(ts[j] - ts[i]) > WINDOW_S:
                break
            pairs.append((cls[i], cls[j]))
            pairs.append((cls[j], cls[i]))
print(f"{len(pairs)//2:,} pares co-ocurrentes en ventana ±{WINDOW_S}s")

In [ ]:
# Matriz de co-ocurrencia (conteo)
comat = np.zeros((9, 9), dtype=int)
for a, b in pairs:
    comat[int(a), int(b)] += 1
np.fill_diagonal(comat, 0)   # diagonal = auto-co-ocurrencia: excluir

labels_9 = [gu.CLASS_NAMES[i] for i in range(9)]
codf = pd.DataFrame(comat, index=labels_9, columns=labels_9)
display(codf)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# Matriz absoluta
sns.heatmap(codf, annot=True, fmt="d", cmap="YlOrRd", linewidths=.4,
            ax=axes[0], cbar_kws={"label":"co-ocurrencias"})
axes[0].set_title(f"Co-ocurrencia absoluta (ventana ±{WINDOW_S}s)")

# Normalizada por mínimo de la fila (P(B|A))
codf_norm = codf.div(codf.sum(axis=1).replace(0, np.nan), axis=0).fillna(0).round(3)
sns.heatmap(codf_norm, annot=True, fmt=".2f", cmap="Blues", linewidths=.4,
            ax=axes[1], cbar_kws={"label":"P(columna | fila)"})
axes[1].set_title("Probabilidad condicional P(B | A)")
for ax in axes: ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.savefig("../outputs/stat_cooccurrence.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# Co-ocurrencia solo para Horn y Siren (vista de peligro)
print("Cuando se detecta Horn, ¿qué clase co-ocurre más?")
horn_row = codf.loc["Horn"].sort_values(ascending=False)
print(horn_row.head(5).to_string())
print()
print("Cuando se detecta Siren:")
if "Siren" in codf.index and codf.loc["Siren"].sum() > 0:
    siren_row = codf.loc["Siren"].sort_values(ascending=False)
    print(siren_row.head(5).to_string())
else:
    print("Siren con pocos eventos co-ocurrentes (n=19 total)")

**Lectura.** Co-ocurrencia de Horn con Speech es esperable (tráfico + radio). Co-ocurrencia Horn+Siren —aunque Siren es escasa— marca momentos de emergencia real. Alta co-ocurrencia de clases OOD entre sí (p.ej. Vibrating + Ring Tone) puede indicar artefactos del modelo (activaciones correladas en el mismo chunk ruidoso).

## Resumen estadístico — tabla maestra por clase

In [ ]:
master = conf_tab.copy().rename(columns={"mean":"conf_mean","std":"conf_std",
                                            "p25":"conf_p25","median":"conf_med","p75":"conf_p75"})
dur2 = (geo.groupby("class_name")["duration_s"]
           .agg(dur_mean="mean", dur_std="std", dur_median="median").loc[CLASS_ORDER])
rate2 = rates.groupby("class_name")[["per_min","per_km"]].mean().loc[CLASS_ORDER].round(3)
master = master.join(dur2).join(rate2)
master.index.name = "clase"
display(master)
master.to_csv("../outputs/stat_master_by_class.csv")
print("guardado outputs/stat_master_by_class.csv")